# Topic Race — Bar Chart Race по топикам Telegram-группы

Этот ноутбук генерирует анимированное видео (вертикальный Reels-формат 1080×1920),
показывающее гонку топиков вашей Telegram-группы по числу постов.

**Что нужно:**
1. Telegram API ключи — получите один раз на [my.telegram.org](https://my.telegram.org)
   (раздел *API development tools* → создайте приложение → скопируйте `api_id` и `api_hash`)
2. Номер телефона, привязанный к Telegram
3. Название группы с включёнными топиками (форумом)

Запустите ячейки по порядку сверху вниз. Весь процесс займёт 5–10 минут.

In [ ]:
#@title ## 1. Установка { display-mode: "form" }
#@markdown Запустите эту ячейку один раз — она установит все зависимости
#@markdown и **автоматически перезагрузит рантайм**.
#@markdown
#@markdown После перезагрузки запускайте ячейки начиная с **шага 2**
#@markdown (эту ячейку повторно запускать НЕ нужно).

import subprocess, sys, os

REPO_DIR = "/content/topic-race"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/dmi3semenov/topic-race.git {REPO_DIR}
else:
    print("Репозиторий уже склонирован.")

# Патч: разрешаем Python 3.10+ (Colab может быть не 3.12)
!sed -i 's/requires-python = ">=3.12"/requires-python = ">=3.10"/' {REPO_DIR}/pyproject.toml

# Патч: Chromium в Colab запускается от root — нужен --no-sandbox
!sed -i 's/p.chromium.launch(headless=True)/p.chromium.launch(headless=True, args=["--no-sandbox", "--disable-dev-shm-usage"])/' {REPO_DIR}/src/topic_race/render_video.py

!pip install -q -e {REPO_DIR}
!pip install -q nest_asyncio

# Устанавливаем Chromium со всеми системными зависимостями (libatk, libglib и т.д.)
!playwright install --with-deps chromium

print("\n✅ Установка завершена! Перезагружаю рантайм...")
print("⏳ После перезагрузки запускайте ячейки начиная с шага 2.")

# Colab не видит новые пакеты без рестарта рантайма
os.kill(os.getpid(), 9)

In [ ]:
#@title ## 2. Настройка { display-mode: "form" }
#@markdown Введите данные Telegram. API ID и название группы — в полях справа.
#@markdown Секретные данные (API Hash и телефон) запросятся отдельно.

import getpass
from pathlib import Path

REPO_DIR = "/content/topic-race"

TG_API_ID = ""  #@param {type:"string"}
GROUP_NAME = ""  #@param {type:"string"}

print("Введите API Hash (из my.telegram.org):")
TG_API_HASH = getpass.getpass("API Hash: ")

print("Введите номер телефона (в формате +7...):")
TG_PHONE = getpass.getpass("Телефон: ")

# Валидация
assert TG_API_ID.strip(), "Укажите TG_API_ID"
assert TG_API_HASH.strip(), "Укажите TG_API_HASH"
assert TG_PHONE.strip(), "Укажите номер телефона"
assert GROUP_NAME.strip(), "Укажите название группы"

# Записываем .env
env_path = Path(REPO_DIR) / ".env"
env_path.write_text(
    f"TG_API_ID={TG_API_ID.strip()}\n"
    f"TG_API_HASH={TG_API_HASH.strip()}\n"
    f"TG_PHONE={TG_PHONE.strip()}\n"
    f"TG_GROUP_NAME={GROUP_NAME.strip()}\n"
)

print(f"\n✅ Настройка сохранена. Группа: «{GROUP_NAME.strip()}»")

In [ ]:
#@title ## 3. Авторизация — запрос кода { display-mode: "form" }
#@markdown Telegram отправит вам код подтверждения.
#@markdown Проверьте Telegram после запуска этой ячейки.

import os
os.chdir("/content/topic-race")

!python -m topic_race.auth request

In [ ]:
#@title ## 4. Авторизация — ввод кода { display-mode: "form" }
#@markdown Введите код из Telegram. Если у вас включена двухфакторная
#@markdown аутентификация, введите также пароль.

OTP_CODE = ""  #@param {type:"string"}
PASSWORD_2FA = ""  #@param {type:"string"}

import os
os.chdir("/content/topic-race")

assert OTP_CODE.strip(), "Введите код из Telegram"

cmd = f"python -m topic_race.auth submit {OTP_CODE.strip()}"
if PASSWORD_2FA.strip():
    cmd += f" --password {PASSWORD_2FA.strip()}"

!{cmd}

In [ ]:
#@title ## 5. Загрузка данных и генерация видео { display-mode: "form" }
#@markdown Настройте параметры видео. Загрузка данных из Telegram может занять
#@markdown несколько минут (зависит от размера группы).

TOP_N = 15  #@param {type:"slider", min:5, max:25, step:1}
DURATION_SEC = 60  #@param {type:"slider", min:30, max:180, step:10}
UPLOAD_AUDIO = False  #@param {type:"boolean"}

import os
os.chdir("/content/topic-race")

import nest_asyncio
nest_asyncio.apply()

from pathlib import Path

# --- Загрузка данных ---
print("📡 Загружаю данные из Telegram...\n")
from topic_race.pipeline import run_sync
group, n_topics, n_new = run_sync(since_days=None, progress=print)
print(f"\n✅ Загружено: {n_topics} топиков, {n_new} новых сообщений\n")

# --- Аудио (опционально) ---
audio_path = None
if UPLOAD_AUDIO:
    from google.colab import files as colab_files
    print("Загрузите аудиофайл (mp3, m4a, wav):")
    uploaded = colab_files.upload()
    if uploaded:
        audio_name = list(uploaded.keys())[0]
        audio_path = Path("/content") / audio_name
        Path(audio_path).write_bytes(uploaded[audio_name])
        print(f"Аудио: {audio_name}")

# --- Рендер видео ---
print("\n🎬 Генерирую видео... (это займёт 1-3 минуты)\n")
from topic_race.render_video import render_reel
from topic_race.config import load_settings

settings = load_settings()

render_kwargs = dict(
    top_n=TOP_N,
    target_duration_sec=float(DURATION_SEC),
    group_name=settings.group_name,
    audio_path=audio_path,
)

# Без аудио: убираем аудио-параметры по умолчанию
if audio_path is None:
    render_kwargs.update(
        audio_start_sec=0,
        audio_tempo=1.0,
        audio_intro_sec=0,
        audio_outro_sec=0,
        audio_fade_out_sec=0,
    )

result = render_reel(**render_kwargs)

print(f"\n✅ Видео готово!")
print(f"   Файл: {result.mp4_path}")
print(f"   Длительность: {result.duration_sec:.0f} сек")
print(f"   Кадров: {result.n_frames}")

In [ ]:
#@title ## 6. Просмотр и скачивание { display-mode: "form" }
#@markdown Видео отобразится ниже. Нажмите кнопку для скачивания.

from IPython.display import Video, display, HTML

# Показываем видео в ноутбуке
display(Video(str(result.mp4_path), embed=True, width=360))

# Кнопка скачивания
try:
    from google.colab import files as colab_files
    print("\nСкачивание начнётся автоматически...")
    colab_files.download(str(result.mp4_path))
except ImportError:
    print(f"\nФайл: {result.mp4_path}")
    print("Скачайте через файловый менеджер слева.")